# LAB 2 for CS4243: Perspective-Aware Virtual Advertisement Overlay
## Notebook 4 — Stage 4: your clip, the Overlay Scorecard, and the generative comparison
> **Introduction.** **Part I** applies the *frozen* pipeline to your one personal clip, compares the two temporal
> variants against your audit annotations, and assembles the Overlay Scorecard. **Part II** runs the generative
> comparison with an open-weight video model on your clip: one attempt with fixed settings and seed, kept and logged, its audit
> frames extracted and scored with the supplied `overlaylab.evaluate --frames`,
> side by side with your classical output on the same frames.
> Question 5 (the reflection) is written from the comparison table and the prompts at the end.

## (Our Code) Reading and execution guide [AI-LANGUAGE] [AI-CODE] [HUMAN-CHECK]

Restart the kernel and run from top to bottom. We analyse both of our personal clips using both classical variants
and compare the original and tuned prompting runs for each clip. The supplied cells analyse the active clip in
`manifests/genai_conditions.json`, while our added sections bring together the results for both clips.

1. **Part I:** our shared setup, classical analysis, full-audit Scorecards and failure figures.
2. **Part II:** the starter export/evaluation example, followed by our comparison of both prompting runs on both clips.
3. **Reflection:** the staff example and Question 5 prompts, supported by our saved comparison reports and figures.

We use our saved classical videos and four raw VACE outputs. The generation code is in
`student_files/stage4/colab_vace_run.ipynb`. We report full-video Scorecard metrics separately from the
selected-frame comparison metrics.

# Part I — Your clip and its Overlay Scorecard
## Suggested investigation structure
1. Describe the clip: board, capture conditions, camera pan / distance change / viewing-angle change, the occlusion
   crossing, with frame ranges (from `manifests/capture_manifest.csv`).
2. Record the audit annotations **before** running anything: 10 corner frames in `manifests/audit_corners.csv`, 5 masks
   (at least 3 on occluded frames) in `manifests/audit_masks.csv`.
   Use `python -m overlaylab.annotate corners | masks | check <clip_id>` (README, "Annotate your clip").
3. Run `python -m overlaylab.run_pipeline --clip <clip_id> --variant independent` and `--variant flow_assisted` with
   `--thresholds outputs/thresholds.json` and your own artwork (`--artwork <name>`), then
   `python -m overlaylab.evaluate --video outputs/videos/<clip_id>_<variant>.mp4 --clip <clip_id> --artwork <name>`.
4. Load the `_stats.json` files written next to the videos and compute corner error, success, jitter, drift and the
   occlusion IoU / F1 / leakage against your audit annotations through `overlaylab.metrics`.
5. Plot the monitor timeline around the occlusion crossing and fill the Scorecard.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import cv2

from overlaylab import data, metrics, visualize as viz
from overlaylab import supplied as sp
from overlaylab.config import notebook_config, replace
from overlaylab.pipelines import load_stats
from overlaylab.temporal import failure_monitor
from overlaylab.evaluate import evaluate_video, evaluate_frames, format_report
import json

cond = data.load_genai_conditions()
CLIP_ID = cond["clip_id"] if not cond["clip_id"].startswith("<") else "my_clip"   # your clip id (data/personal/<clip_id>/), also set in manifests/genai_conditions.json
ARTWORK = Path(cond["artwork"]).stem if "<" not in cond["artwork"] else None
# Set your own artwork in genai_conditions.json; never silently substitute the supplied ad.
if ARTWORK is None: print("Set your own artwork in manifests/genai_conditions.json before Stage 4.")
from overlaylab.config import PipelineConfig
freeze_path = ROOT / "outputs" / "thresholds.json"
frozen = PipelineConfig.from_json(freeze_path.read_text()) if freeze_path.exists() else PipelineConfig()
SUCCESS_THRESHOLD = frozen.localisation.success_threshold
video, reference_path = data.clip_paths(CLIP_ID)
print("clip:", CLIP_ID, "| video exists:", video.exists(), "| artwork:", ARTWORK)
audit_c = data.load_audit_corners(CLIP_ID) if (data.MANIFEST_DIR / "audit_corners.csv").exists() else {}
audit_m = data.load_audit_masks(CLIP_ID) if (data.MANIFEST_DIR / "audit_masks.csv").exists() else {}
print(len(audit_c), "audit corner frames", sorted(audit_c), "|", len(audit_m), "audit masks", sorted(audit_m))
stats_path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_flow_assisted_stats.json"
if stats_path.exists() and audit_c:
    st = load_stats(stats_path)
    scale = (st["resize_width"] or sp.video_info(video)["width"]) / sp.video_info(video)["width"]
    errs = {f: metrics.corner_error(np.asarray(st["corners"][f]) / scale, audit_c[f]) for f in audit_c if f < len(st["corners"])}
    print("corner errors on audit frames (%):", {f: round(e * 100, 2) for f, e in errs.items()})
    viz.plot_monitor_timeline(st["stats"], [tuple(x) for x in st["flags"]], errors=errs, title=CLIP_ID); plt.show()
else:
    print("run the pipeline on your clip first:", stats_path.name, "not found (or no audit corners yet)")

## (Our Code) Shared setup for both personal clips [AI-CODE] [HUMAN-CHECK]

We define our clip/artwork mapping, representative and failure frames, and output roots here. We load the invocation paths from the preserved run index.

In [ ]:
"""Complete the two personal clips' classical evaluation without retuning."""

import copy
import csv
import hashlib
import shutil
from collections import Counter
from IPython.display import display, Markdown, Image

from overlaylab import metrics as M
from overlaylab.compositing import composite, occlusion_mask
from overlaylab.geometry import dlt_homography, warp_artwork
from overlaylab.localisation import artwork_to_frame, localise_board
from overlaylab.report import mask_scores, summarise_tracking

CODE = ROOT
HERE = CODE / "student_files" / "stage4"
OUT = CODE / "outputs" / "evaluation"
FIGURES = HERE
CLASSICAL_CLIPS = {"stage4_sean": ("sean_explore", 300, 270),
                   "stage4_ruijie": ("ruijie_orbit", 0, 255)}

RECORDS = HERE / "run_records"
RUN_ENTRIES = json.loads((RECORDS / "index.json").read_text())

## (Our Code) Classical evidence helpers [AI-CODE] [HUMAN-CHECK]

We reconstruct geometry from our saved predictions and use these helpers to draw diagnostic matches and save the source/overlay/failure figures and monitor timelines. We use audit corners for scoring and illustration.

In [ ]:
def finite(value, spec=".3f"):
    return format(value, spec) if value is not None and np.isfinite(value) else "n/a"

def percent(value):
    return f"{value:.3%}" if value is not None and np.isfinite(value) else "n/a"

def frames_at(path, indices):
    return {i: f for i, f in sp.iter_video(path, stop=max(indices)+1) if i in set(indices)}

def saved_h(stats, frame, reference):
    q = np.asarray(stats["corners"][frame], float)
    if stats["status"][frame] != "ok" or stats["flags"][frame][0] == "HIDE" or not np.isfinite(q).all():
        return None
    H, _, ok = dlt_homography(sp.rect_corners(reference.shape[1], reference.shape[0]), q)
    return H if ok else None

def episodes(flags):
    result, start = [], None
    for i, (flag, _) in enumerate(flags + [["END", "none"]]):
        if flag == "HIDE" and start is None:
            start = i
        elif flag != "HIDE" and start is not None:
            result.append([start, i-1])
            start = None
    return result

def match_picture(reference, frame, diag, matches):
    height = 240
    a = cv2.resize(reference, (round(reference.shape[1]*height/reference.shape[0]), height))
    b = cv2.resize(frame, (round(frame.shape[1]*height/frame.shape[0]), height))
    image = np.concatenate([a, b], axis=1).copy()
    for ri, fi in matches[:30]:
        x = diag["keypoints_ref"][ri] * height/reference.shape[0]
        y = diag["keypoints_frame"][fi] * height/frame.shape[0] + [a.shape[1], 0]
        cv2.line(image, tuple(np.round(x).astype(int)), tuple(np.round(y).astype(int)), (1., .25, .1), 1, cv2.LINE_AA)
    return image

def make_figures(clip, stats, gt, sources, overlays, reference, cfg, representative, failure):
    flow = stats["flow_assisted"]
    H = saved_h(flow, representative, reference)
    difference = np.zeros(sources[representative].shape[:2], np.float32)
    predicted = np.zeros_like(difference, bool)
    if H is not None:
        predicted, difference, _ = occlusion_mask(sources[representative], H, reference, cfg.occlusion)
    _, _, pairs, _, diag = localise_board(reference, sources[failure], cfg.localisation)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for row, frame in enumerate([representative, failure]):
        axes[row, 0].imshow(sources[frame]); axes[row, 0].set_title(f"Source frame {frame}")
        axes[row, 1].imshow(overlays[frame]); axes[row, 1].set_title(f"Flow-assisted: {flow['flags'][frame][0]}")
        for col in [0, 1]:
            q = gt[frame]
            axes[row, col].plot(*np.vstack([q, q[0]]).T, color="lime", linewidth=1)
    axes[0, 2].imshow(difference, cmap="inferno", vmin=0, vmax=.5)
    if predicted.any():
        axes[0, 2].contour(predicted, levels=[.5], colors="cyan", linewidths=.7)
    axes[0, 2].set_title("Compensated difference / cyan predicted mask")
    axes[1, 2].imshow(match_picture(reference, sources[failure], diag, pairs))
    axes[1, 2].set_title(f"Failed fixed localiser: {diag['n_matches']} matches, {diag['n_inliers']} inliers")
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"{clip}: full-resolution source and frozen classical evidence")
    fig.tight_layout()
    fig.savefig(FIGURES / f"{clip}_classical_evidence.png", dpi=140)
    plt.close(fig)
    fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
    colors = {"independent":"tab:blue", "flow_assisted":"tab:orange"}
    for row, (variant, st) in enumerate(stats.items()):
        x = np.arange(st["n_frames"])
        axes[0].plot(x, [s["inlier_count"] for s in st["stats"]], color=colors[variant], label=variant, alpha=.8)
        errors = [(f, M.corner_error(np.asarray(st["corners"][f]), q)*100) for f, q in gt.items()
                  if st["status"][f] == "ok" and st["flags"][f][0] != "HIDE"]
        axes[1].scatter([a for a,b in errors], [b for a,b in errors], color=colors[variant], s=25, label=variant)
        flags = np.array([0 if z[0] == "OK" else 1 if z[0] == "REINIT" else 2 for z in st["flags"]])
        axes[2].imshow(flags[None], aspect="auto", extent=[-.5,449.5,row-.4,row+.4],
                       cmap=plt.matplotlib.colors.ListedColormap(["#55aa66","#e5b54d","#cc6677"]), vmin=0, vmax=2)
    axes[0].axhline(cfg.track.monitor.min_inlier_count, color="black", linestyle="--", linewidth=.8)
    axes[0].set_ylabel("Inliers"); axes[0].legend(loc="upper right")
    axes[1].axhline(cfg.localisation.success_threshold*100, color="black", linestyle="--", linewidth=.8)
    axes[1].set_ylabel("Finite audit error (%)"); axes[1].legend(loc="upper right")
    axes[2].set_yticks([0,1], ["independent","flow_assisted"])
    axes[2].set_xlabel("Prepared source frame index (green OK / yellow REINIT / red HIDE)")
    for ax in axes:
        a, b = (233,313) if clip == "stage4_sean" else (340,395)
        ax.axvspan(a, b, color="skyblue", alpha=.15)
        ax.axvline(representative, color="grey", linestyle=":", alpha=.7)
        ax.axvline(failure, color="purple", linestyle=":", alpha=.7)
        ax.grid(alpha=.15)
    fig.suptitle(f"{clip}: frozen monitor and available audit predictions (unavailable errors omitted)")
    fig.tight_layout(); fig.savefig(FIGURES / f"{clip}_classical_timeline.png", dpi=140); plt.close(fig)
    truth = data.load_audit_masks(clip).get(representative)
    return {"failure_frame":failure,"representative_frame":representative,
            "failure_localiser":{"matches":diag["n_matches"],"inliers":diag["n_inliers"],
                                 "status":bool(diag["status"]),"median_residual":diag["median_residual"]},
            "representative_mean_difference_inside_board":float(difference[predicted].mean()) if predicted.any() else None,
            "strong_fraction_of_gt_occluder":float((difference[truth] >= cfg.occlusion.strong).mean()) if truth is not None and truth.any() else None,
            "predicted_fraction_of_gt_occluder":float(predicted[truth].mean()) if truth is not None and truth.any() else None}

## Classical evaluation on the audit frames
`overlaylab.evaluate` scores both variants of your clip on your audit corners and masks (placement, fidelity, temporal,
preservation, occlusion incl. the visible-board coverage). These are the numbers for the Scorecard.

In [ ]:
classical = {}
for variant in ("independent", "flow_assisted"):
    path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_{variant}.mp4"
    if path.exists() and audit_c and ARTWORK:
        classical[variant] = evaluate_video(path, video, data.load_artwork(ARTWORK), gt_corners=audit_c, gt_masks=audit_m, success_threshold=SUCCESS_THRESHOLD, verbose=False)
        print(format_report(classical[variant]), "\n")
        folder = ROOT / "outputs" / "evaluation"
        folder.mkdir(parents=True, exist_ok=True)
        (folder / f"{CLIP_ID}_{variant}_report.json").write_text(json.dumps(classical[variant], indent=2, default=float))
    else:
        print("missing:", path.name)

## (Our Code) Full-audit classical analysis and Scorecard calculation [AI-CODE] [HUMAN-CHECK]

We use the saved full-video reports for both clips, check that the runs use our frozen settings, and compute tracking and mask summaries. We also extract the classical comparison frames into `student_files/stage4/comparison_frames/`. We reuse their identical-frame evaluation reports in Part II.

In [ ]:
def analyse_classical():
    cfg = PipelineConfig.from_json((CODE / "outputs/thresholds.json").read_text())
    OUT.mkdir(parents=True, exist_ok=True)
    score_dir = CODE / "outputs/scorecards"; score_dir.mkdir(parents=True, exist_ok=True)
    captures = {r["clip_id"]:r for r in sp.read_csv(CODE / "manifests/capture_manifest.csv")}
    all_results = {}
    for clip, (art_name, representative, failure) in CLASSICAL_CLIPS.items():
        video, ref_path = data.clip_paths(clip)
        reference, art = sp.read_image(ref_path), data.load_artwork(art_name)
        cfg.localisation.warp.reference_shape = tuple(reference.shape[:2])
        gt, masks = data.load_audit_corners(clip), data.load_audit_masks(clip)
        needed = sorted({0,representative,failure,*masks})
        source = frames_at(video, needed)
        _, _, _, _, init = localise_board(reference, source[0], cfg.localisation)
        init_record = {k:init[k] for k in ["n_keypoints_ref","n_keypoints_frame","n_matches","n_inliers","inlier_ratio","median_residual"]}
        result = {"clip_id":clip,"processing_resolution":[1280,720],"audit_frames":sorted(gt),
                  "mask_frames":sorted(masks),"initialisation":init_record,"variants":{},"capture":captures[clip]}
        stats = {}
        for variant in ["independent","flow_assisted"]:
            path = CODE / f"outputs/videos/{clip}_{variant}_stats.json"
            st = json.loads(path.read_text()); stats[variant] = st
            output = path.with_name(f"{clip}_{variant}.mp4")
            assert sp.video_info(output) == {"frames":450,"fps":30.,"width":1280,"height":720}
            assert st["n_frames"] == 450 and st["resize_width"] is None
            expected = copy.deepcopy(cfg); expected.track.variant = variant
            assert st["frozen_config"] == expected.thresholds()
            assert st["request"]["source_sha256"] == hashlib.sha256(video.read_bytes()).hexdigest()
            assert st["request"]["artwork_sha256"] == hashlib.sha256(np.ascontiguousarray(art).tobytes()).hexdigest()
            track = summarise_tracking(st, gt, cfg.localisation.success_threshold, 1280)
            mask_stats = copy.deepcopy(st)
            mask_stats["mask_dir"] = str(path.with_name(f"{clip}_{variant}_masks"))
            score = mask_scores(mask_stats, masks, sorted(masks))
            alpha_with, alpha_without, per_mask = [], [], []
            for f, truth in masks.items():
                if not truth.any():
                    continue
                H = saved_h(st, f, reference)
                masked_alpha = baseline_alpha = np.zeros(truth.shape, np.float32)
                if H is not None:
                    pred, _, _ = occlusion_mask(source[f], H, reference, cfg.occlusion)
                    Ha = artwork_to_frame(art, H, cfg.localisation)
                    warped, coverage = warp_artwork(art, Ha, source[f].shape)
                    _, masked_alpha = composite(source[f], warped, coverage, pred, cfg.composite)
                    _, baseline_alpha = composite(source[f], warped, coverage, np.zeros(truth.shape,bool), cfg.composite)
                with_mask, without_mask = M.leakage(masked_alpha,truth), M.leakage(baseline_alpha,truth)
                alpha_with.append(with_mask); alpha_without.append(without_mask)
                per_mask.append({"frame":f,"hidden":H is None,"alpha_leakage":with_mask,"no_occlusion_alpha_leakage":without_mask})
            rendered = json.loads((OUT / f"{clip}_{variant}_report.json").read_text())
            conditions = json.loads((CODE / f"outputs/genai/{clip}/conditions.json").read_text())
            comparison = conditions["frames"]
            chosen = sorted({comparison[k] for k in ["clean","occluded","perspective"]} | set(comparison["pair"]))
            frame_dir = HERE / "comparison_frames" / clip / f"classical_{variant}"
            frame_dir.mkdir(parents=True, exist_ok=True)
            for f, image in frames_at(output, chosen).items():
                sp.write_image(frame_dir / f"frame_{f:06d}.png", image)
            baseline_report = evaluate_frames([frame_dir / f"frame_{f:06d}.png" for f in chosen],
                video, art, gt_corners=gt, gt_masks=masks,
                success_threshold=cfg.localisation.success_threshold, verbose=False)
            (OUT / f"{clip}_{variant}_frames_report.json").write_text(json.dumps(baseline_report,indent=2)+"\n")
            result["variants"][variant] = {"tracking":track,"mask_scores":score,"rendered":rendered,
                "time_s":sum(st["timing"].values()),"tracking_s":st["timing"]["track_s"],
                "composite_s":st["timing"]["composite_s"],"hide_episodes":episodes(st["flags"]),
                "triggers":dict(Counter(x[1] for x in st["flags"])),
                "flow_frames":sum(bool(z) for z in st["flow"]),
                "comparison_baseline":baseline_report,
                "direct_alpha_leakage":float(np.mean(alpha_with)),
                "baseline_alpha_leakage":float(np.mean(alpha_without)),"alpha_mask_frames":per_mask}
        overlay = frames_at(CODE / f"outputs/videos/{clip}_flow_assisted.mp4", [representative,failure])
        result["evidence"] = make_figures(clip, stats, gt, source, overlay, reference, cfg, representative, failure)
        q = gt[0];result["first_frame_board_fraction"] = float(abs(np.dot(q[:,0],np.roll(q[:,1],-1))-np.dot(q[:,1],np.roll(q[:,0],-1)))/2/(1280*720))
        fields = {"Clip":f"{clip}, 15 s, 1280 x 720 at 30 fps. {captures[clip]['board_description']}. First-frame annotated area {result['first_frame_board_fraction']:.2%}. Event ranges: {captures[clip]['event_frame_ranges']}.",
          "Initialisation":f"Descriptor-valid keypoints reference/frame 0: {init['n_keypoints_ref']}/{init['n_keypoints_frame']}. Matches {init['n_matches']}, inliers {init['n_inliers']} ({init['inlier_ratio']:.3f}), median residual {init['median_residual']:.3f} px.",
          "Geometry":"", "Temporal":"", "Occlusion":"", "Monitor":"",
          "Description":("The overlay is available on only four of ten audit frames. Frame 300 paints much of the purple foreground object, while frame 270 is hidden after localisation failure." if clip=='stage4_sean' else "The overlay is available on three of ten audit frames. The strongest-perspective frame and all five mask frames are hidden, so zero leakage is explained by absent overlays."),
          "Uncertainty":"All settings and seed 0 were transferred without personal-clip tuning. Dimensions and event ranges are approximate. Manual corners behind occluders are uncertain. Tracker jitter uses sparse GT motion interpolation. Rendered placement uses SIFT re-localisation and differs from saved-corner error. No full board exit/re-entry occurs, so re-entry recovery is not assessed."}
        for variant, v in result['variants'].items():
            t, m, e = v['tracking'],v['mask_scores'],v['rendered']
            fields['Geometry'] += f"{variant}: median saved-corner error {t['median_error']:.3%} on {t['n']-t['failed']}/10 finite frames, success {t['success']:.0%} at 2%, unavailable {t['failed']}/10. "
            fields['Temporal'] += f"{variant}: tracker jitter {finite(t['jitter_px'])} px, drift at least 16 frames after reinitialisation {percent(t['drift_error'])} on {t['drift_frames']} finite audit frames, longest localisation interval {t['longest_run']} frames. "
            fields['Occlusion'] += f"{variant}: IoU {finite(m['iou'])}, F1 {finite(m['f1'])} over {m['mask_frames_scored']}/3 nonempty masks, hidden {len(m['mask_frames_hidden'])}, missing {len(m['mask_frames_missing'])}. Finished-video leakage {e['occlusion']['leakage']:.2%} over all 3 nonempty masks. Coverage {percent(e['occlusion']['coverage_visible'])} on {e['occlusion']['n_coverage_frames']} overlay-present mask frames. Direct alpha leakage with/without masking {v['direct_alpha_leakage']:.2%}/{v['baseline_alpha_leakage']:.2%}, same saved geometry and HIDE policy. "
            fields['Monitor'] += f"{variant}: {t['hide']}/450 HIDE frames, {len(v['hide_episodes'])} HIDE episodes, {t['reinit']} REINIT flags, {t['localisations']} localisation calls, {t['frames_tracked']}/450 tracked. Pipeline time {v['time_s']:.2f} s. "
        text = '\n'.join([f"# Overlay Scorecard — {clip}",'']+[f"**{k}**: {v.rstrip()}\n" for k,v in fields.items()])
        text += f"\nEvidence and monitor: [source/overlay/failure](../../student_files/stage4/{clip}_classical_evidence.png), [timeline](../../student_files/stage4/{clip}_classical_timeline.png).\n"
        (score_dir / f"{clip}_scorecard.md").write_text(text.rstrip()+'\n')
        all_results[clip] = result
        print(clip, 'classical analysis and Scorecard saved.')
    (FIGURES / 'classical_summary.json').write_text(json.dumps(all_results,indent=2)+'\n')
    return all_results

## Overlay Scorecard
Fill the dictionary from your measurements and write the Markdown scorecard to `outputs/scorecards/<clip_id>_scorecard.md`.
Every number must come from `overlaylab.metrics` or `overlaylab.evaluate`; the description and the uncertainty statement are yours.

In [ ]:
def write_scorecard(clip_id, fields):
    lines = [f"# Overlay Scorecard — {clip_id}", ""]
    for k, v in fields.items():
        lines.append(f"**{k}**: {v}  ")
    (ROOT / "outputs" / "scorecards").mkdir(parents=True, exist_ok=True)
    (ROOT / "outputs" / "scorecards" / f"{clip_id}_scorecard.md").write_text("\n".join(lines))
    return "\n".join(lines)

scorecard = {
    "Clip": "<conditions, resolution, fps, board size, event frame ranges>",
    "Initialisation": "<keypoints on reference / frame 0, matches, inliers (ratio), median residual>",
    "Geometry": "<median corner error and success fraction for both variants>",
    "Temporal": "<jitter and drift for both variants; recovery after re-entry>",
    "Occlusion": "<IoU, F1, leakage on masked frames; visible-board coverage; baseline leakage>",
    "Monitor": "<timeline: flags, frames and triggering statistics>",
    "Description": "<one evidence-grounded paragraph>",
    "Uncertainty": "<annotation uncertainty, thresholds transferred from the common clip, ...>",
}
# YOUR CODE HERE — replace the placeholders with measured values, then write the file
print(write_scorecard(CLIP_ID, scorecard))

## (Our Code) Completed Scorecards and evidence for both clips — modified copy including retained starter code [AI-CODE] [HUMAN-CHECK]

We retain the starter Scorecard writer and complete the measurements for both clips. After the supplied cell writes its template, we save our completed Scorecards, display both summaries and show the four classical report figures.

In [ ]:
def write_scorecard(clip_id, fields):
    lines = [f"# Overlay Scorecard — {clip_id}", ""]
    for k, v in fields.items():
        lines.append(f"**{k}**: {v}  ")
    (ROOT / "outputs" / "scorecards").mkdir(parents=True, exist_ok=True)
    (ROOT / "outputs" / "scorecards" / f"{clip_id}_scorecard.md").write_text("\n".join(lines))
    return "\n".join(lines)

personal_classical = analyse_classical()
scorecard_path = ROOT / "outputs/scorecards" / f"{CLIP_ID}_scorecard.md"
scorecard_text = scorecard_path.read_text()
scorecard = {}
for line in scorecard_text.splitlines():
    if line.startswith("**"):
        key, value = line[2:].split("**: ", 1)
        scorecard[key] = value
# YOUR CODE HERE — replace the placeholders with measured values, then write the file
print(scorecard_text)

for personal_clip, result in personal_classical.items():
    print("\n", personal_clip, "| 450 frames | 1280 x 720 | 30 fps")
    print("Initialisation:", result["initialisation"])
    for variant, measures in result["variants"].items():
        tracking, masks = measures["tracking"], measures["mask_scores"]
        print(variant, "success", tracking["success"], "finite audit frames", 10-tracking["failed"],
              "tracked", tracking["frames_tracked"], "/450", "HIDE", tracking["hide"],
              "jitter px", tracking["jitter_px"], "mask scores", masks, "time s", measures["time_s"])
    scorecard_path = ROOT / "outputs" / "scorecards" / f"{personal_clip}_scorecard.md"
    print("Scorecard:", scorecard_path)
    display(Markdown(scorecard_path.read_text().split("\nEvidence and monitor:")[0]))
    for variant, measures in result["variants"].items():
        baseline = measures["comparison_baseline"]
        print("Fixed comparison baseline:", variant, "source indices", sorted(baseline["frames"]), "placement success", baseline["placement"]["success"])
print("Full classical metrics saved to student_files/stage4/classical_summary.json")

from IPython.display import display, Image
for personal_clip in personal_classical:
    display(Image(filename=str(ROOT / "student_files" / "stage4" / f"{personal_clip}_classical_evidence.png")))
    display(Image(filename=str(ROOT / "student_files" / "stage4" / f"{personal_clip}_classical_timeline.png")))

# Part II — The generative comparison on your clip
## The prompt

The model receives your clip (or your cut of it), your artwork and a one-line instruction (plus your pipeline's board masks if it
needs a mask). This asks whether it can carry out the intended edit without explicit geometric or preservation
instructions.

Use **one open-weight video model** that returns the edited video itself (inference scripts are allowed; the edit itself must be generated pixels) and keep it, its version and
its settings (`model` in `manifests/genai_conditions.json`). Use a model your hardware supports, for example
MiniMax-H3 [MiniMax, 2026, https://huggingface.co/MiniMaxAI/MiniMax-H3], JoyAI-Video-Edit [JD Open Source, 2026,
https://github.com/jd-opensource/JoyAI-Video-Edit], LTX-2.3 [Lightricks, 2026] or Wan2.1-VACE-14B [Wan, arXiv:2503.20314;
VACE, Jiang et al., arXiv:2503.07598]; other open-weight models are equally welcome, and the grade rests on the comparison
rather than on the model. Wan2.1-VACE-1.3B (Apache 2.0; staff trial: 14 GB GPU memory and 13 min
for a 5.5 s cut at 480p) is a smaller option; memory use depends on settings. Cite the model you use. Give the model your
whole clip; if it does not accept a clip this long, **cut it**: a shorter interval that contains all of the frames below,
or consecutive segments run with the same settings whose outputs you join in order. **One attempt**, including any duration-limited segments, with fixed settings and one seed
(`model.seed`), its output kept as returned and logged in `manifests/genai_log.csv`, `[AI-GENERATION]` on every generated
pixel.

The comparison is scored on these audit frames: one **clean** frame (nothing in front of the board), one **occluded** frame (a mask
frame), the frame with the **strongest perspective**, and one consecutive **pair** (f, f+1) for the temporal terms.
`data/genai_oracle/` holds the same edit run by the staff on the common clip with MiniMax-H3 (33 B parameters) and its
report: an example of what a much larger model does, not a target.

## 1. Choose the frames (and a cut if your model needs one), export the inputs
Fill `frames` in `manifests/genai_conditions.json` **before the first generation**. Leave `model.interval` empty to give the
model the whole clip, or set it to the first and last frame of your cut. The cell exports to
`outputs/genai/inputs/`: the clip or cut as `interval.mp4` (source frame rate) and `masks/mask_XXXXXX.png`, the board quadrilateral your *frozen* pipeline tracked on every frame of the
interval (white = board), for models that take a mask. The audit annotations are never given to the model.

## (Our Code) Paths for the executable starter example [AI-CODE] [HUMAN-CHECK]

The starter cells use generic `outputs/genai/` paths. We temporarily place the active clip's preserved revised-prompt video there while those cells run. After the final comparison, we remove the newly created generic exports, extracts and duplicate reports. Our inputs and raw runs remain in their clip-specific folders.

In [ ]:
active_entry = next(entry for entry in RUN_ENTRIES
                    if entry["clip_id"] == CLIP_ID and entry["role"] == "corrected")
starter_genai = CODE / "outputs/genai"
starter_indices = sorted({cond["frames"][key] for key in ("clean", "occluded", "perspective")}
                        | set(cond["frames"]["pair"]))
starter_paths = [starter_genai / "inputs", starter_genai / "output.mp4",
                 *(starter_genai / f"frame_{frame:06d}.png" for frame in starter_indices),
                 *(starter_genai / f"classical_{variant}" for variant in ("independent", "flow_assisted")),
                 *(OUT / f"classical_{variant}_frames_report.json"
                   for variant in ("independent", "flow_assisted"))]
starter_created_paths = [item for item in starter_paths if not item.exists()]
if not (starter_genai / "output.mp4").exists():
    shutil.copyfile(CODE / active_entry["output"], starter_genai / "output.mp4")
print("Starter cells use", CLIP_ID, "and its revised-prompt output.")
print("The final both-person analysis saves reports directly in outputs/evaluation/.")

In [ ]:
from overlaylab.pipelines import load_stats
cond = data.load_genai_conditions(); fr = cond["frames"]; iv = cond["model"]["interval"]
assert not cond["clip_id"].startswith("<"), "set clip_id in manifests/genai_conditions.json"
assert all(isinstance(fr[k], int) for k in ("clean", "occluded", "perspective")), "fill frames.clean / occluded / perspective in genai_conditions.json"
assert isinstance(fr["pair"], list) and len(fr["pair"]) == 2 and all(isinstance(x, int) for x in fr["pair"]) and fr["pair"][1] == fr["pair"][0] + 1, "frames.pair must be two consecutive frame indices"
assert fr["occluded"] in audit_m and fr["occluded"] in audit_c, "the occluded frame needs both an audit mask and audit corners"
assert fr["clean"] in audit_c and fr["perspective"] in audit_c, "clean and perspective frames must be audit corner frames"
assert len({fr["clean"], fr["occluded"], fr["perspective"]}) == 3, "choose three distinct audit frames"
assert ARTWORK is not None, "set your own artwork first"
chosen = sorted({fr["clean"], fr["occluded"], fr["perspective"], *fr["pair"]})
info = sp.video_info(video)
iv = {"start": iv["start"] if isinstance(iv.get("start"), int) else 0,
      "end": iv["end"] if isinstance(iv.get("end"), int) else info["frames"] - 1}       # empty = the whole clip
assert 0 <= iv["start"] <= min(chosen) and max(chosen) <= iv["end"] < info["frames"], f"the cut {iv['start']}..{iv['end']} must contain the frames {chosen}"
print("model:", cond["model"].get("name"), "| interval", iv["start"], "-", iv["end"], f"({(iv['end'] - iv['start'] + 1) / info['fps']:.1f} s)", "| audit frames", chosen)
inputs = ROOT / "outputs" / "genai" / "inputs"; (inputs / "masks").mkdir(parents=True, exist_ok=True)
stats_path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_flow_assisted_stats.json"
stats = load_stats(stats_path) if stats_path.exists() else None
if stats is None:
    print("  no", stats_path.name, "- run the frozen pipeline first (Part I) if your model needs board masks")
for stale in (inputs / "masks").glob("mask_*.png"): stale.unlink()
first = None
with sp.VideoWriter(inputs / "interval.mp4", info["fps"], (info["width"], info["height"])) as w:
    for i, f in sp.iter_video(video, start=iv["start"], stop=iv["end"] + 1):
        w.write(f)
        if first is None:
            first = (i, f)
        if stats is not None:
            c = stats["corners"][i]
            c = np.asarray(c, float) * info["width"] / (stats.get("resize_width") or info["width"])
            q = np.asarray(c, float) if c is not None else None
            ok = q is not None and q.shape == (4, 2) and np.all(np.isfinite(q))      # frames the pipeline lost carry NaN corners
            m = sp.quad_mask(f.shape[:2], q) if ok else np.zeros(f.shape[:2], bool)
            sp.write_mask(inputs / "masks" / f"mask_{i:06d}.png", m)
print("exported interval.mp4 and", len(list((inputs / "masks").glob("*.png"))), "masks to", inputs)
print("\ninputs:", cond["prompt"]["inputs"], "\nprompt:", cond["prompt"]["template"])
print("\nSave the model's video as outputs/genai/output.mp4, then run the next cell.")

## 2. Extract the audit frames from every output video
A video model resamples the clip to its own frame rate (16 fps for Wan, 24 fps for MiniMax-H3) and may re-time the motion.
Output frame *k* is taken to show source time `start + k · fps_source / fps_output`; the cell writes the output frame closest
to each audit frame's source time, resized to your clip's resolution, as `frame_XXXXXX.png` with the **original index**.
Set `model.output_fps` in `genai_conditions.json` if the container reports the wrong frame rate. If you split the clip into
segments, join the outputs in order into one `output.mp4` first. If the model changed the
timing, these frames will not line up with your clip, and the evaluator will count that as an error. Report it.

In [ ]:
gdir = ROOT / "outputs" / "genai"; mp4 = gdir / "output.mp4"
if not mp4.exists():
    print("no outputs/genai/output.mp4 yet")
else:
    oinfo = sp.video_info(mp4); ofps = cond["model"].get("output_fps") or oinfo["fps"]
    assert ofps > 0, "output fps must be positive"
    for stale in gdir.glob("frame_*.png"): stale.unlink()
    want = {f: int(round((f - iv["start"]) * ofps / info["fps"])) for f in chosen}
    if len(set(want.values())) != len(want):
        raise ValueError("output frame rate maps distinct selected source frames to one output frame; choose a consecutive pair that maps to two distinct frames before generation")
    got = {k: f for k, f in sp.iter_video(mp4) if k in set(want.values())}
    for f, k in want.items():
        if k not in got:
            raise ValueError(f"output has no frame {k} for required source frame {f}; do not score an incomplete comparison")
        out = cv2.resize(sp.to_uint8(got[k]), (info["width"], info["height"]), interpolation=cv2.INTER_CUBIC)
        sp.write_image(gdir / f"frame_{f:06d}.png", out)
    print(f"output.mp4: {oinfo['frames']} frames at {ofps:g} fps, {oinfo['width']}x{oinfo['height']} -> audit frames", {f: k for f, k in want.items()})

## 3. Log the run before looking at the metrics
One attempt in `manifests/genai_log.csv` (one row per segment/invocation if segmented; log technical retries too) (copy the template; the fields are `cond["log_fields"]`). Record generated pixels with
`[AI-GENERATION]`; state the model, visible version, date, settings, the seed, the video you gave it (whole clip, cut, or
joined segments), the output frame rate, the prompt, time and compute.

In [ ]:
import csv, shutil
log = ROOT / "manifests" / "genai_log.csv"
if not log.exists():
    shutil.copy(ROOT / "manifests" / "genai_log_template.csv", log)
rows = list(csv.DictReader(open(log)))
fields = list(csv.DictReader(open(log)).fieldnames)
print(len(rows), "invocation(s) logged for one attempt (multiple rows are allowed for segments/technical retries)")
missing_fields = [f for f in cond["log_fields"] if f not in fields]
print("fields:", fields, "| missing:", missing_fields or "none")
for r in rows:
    if not r.get("model", "").strip() or not r.get("seed", "").strip():
        print("  WARNING: the log row does not declare model / seed")

## (Our Code) Invocation summary for both clips and both prompts [AI-CODE] [HUMAN-CHECK]

We use the log loaded above to compare the original and tuned prompts, seeds, intervals, processing times and compute for both clips. The preserved run index identifies our four outputs.

In [ ]:
for personal_clip in CLASSICAL_CLIPS:
    print("\n", personal_clip)
    for entry in RUN_ENTRIES:
        if entry["clip_id"] != personal_clip:
            continue
        record = json.loads((CODE / entry["records"] / "conditions.json").read_text())
        invocation = next(row for row in rows
                          if json.loads(row["notes"]).get("invocation_id") == entry["invocation_id"])
        label = "original prompt" if entry["role"] == "original" else "tuned prompt"
        print(label, "| seed", record["model"]["seed"],
              "| interval", record["model"]["interval"]["start"], "-", record["model"]["interval"]["end"],
              "| time", invocation["processing_time_s"], "s |", invocation["compute"])
        print("Prompt:", invocation["prompt"])

## 4. Evaluate the generative output and the classical output on the same frames
`overlaylab.evaluate --frames` re-localises the artwork in each generated frame with SIFT + RANSAC, compares the rectified
board with the artwork, checks what changed outside the board, measures leakage and visible-board coverage on the mask
frame, and the flicker / warping error / jitter on the consecutive pair. Both classical videos are extracted and scored with
`evaluate_frames` on the **same selected indices and full audit dictionary**, so every row of the table is computed on identical inputs.

In [ ]:
assert ARTWORK is not None, "set your own artwork"
art = data.load_artwork(ARTWORK)
# Both methods use the identical selected source indices and the full audit dictionary.
# This also gives both the same interpolation on the consecutive pair.
reports = {}
for variant in ("independent", "flow_assisted"):
    path = ROOT / "outputs" / "videos" / f"{CLIP_ID}_{variant}.mp4"
    assert path.exists(), f"run {variant} on your clip first"
    folder = ROOT / "outputs" / "genai" / f"classical_{variant}"
    folder.mkdir(parents=True, exist_ok=True)
    for stale in folder.glob("frame_*.png"): stale.unlink()
    for i, frame in sp.iter_video(path, stop=max(chosen) + 1):
        if i in chosen: sp.write_image(folder / f"frame_{i:06d}.png", frame)
    paths = [folder / f"frame_{f:06d}.png" for f in chosen]
    assert all(p.exists() for p in paths), "classical output is incomplete"
    reports[f"classical_{variant}"] = evaluate_frames(paths, video, art, gt_corners=audit_c, gt_masks=audit_m,
        success_threshold=SUCCESS_THRESHOLD, verbose=False)
paths = [ROOT / "outputs" / "genai" / f"frame_{f:06d}.png" for f in chosen]
assert all(p.exists() for p in paths), "extract every selected generative frame first"
reports["generative"] = evaluate_frames(paths, video, art, gt_corners=audit_c, gt_masks=audit_m,
    success_threshold=SUCCESS_THRESHOLD, verbose=False)
for name, rep in reports.items(): print(name, format_report(rep), "\n")
(ROOT / "outputs" / "evaluation").mkdir(exist_ok=True)
(ROOT / "outputs" / "evaluation" / "stage4_genai_reports.json").write_text(json.dumps(reports, indent=2, default=float))

for name, rep in reports.items():
    fname = "genai_frames_report.json" if name == "generative" else f"{name}_frames_report.json"
    (ROOT / "outputs" / "evaluation" / fname).write_text(json.dumps(rep, indent=2, default=float))


## 5. Comparison table
Placement, fidelity, temporal consistency, preservation and occlusion for the generative output next to the classical
output on the same frames, plus processing time and compute from your log. The background PSNR is the signature
of what the system did: pixels *composited by code* leave the background unchanged up to compression (about 40 dB: your
pipeline), a *re-rendered* frame scores about 20-30 dB (typically the generative model).

In [ ]:
# YOUR CODE HERE — complete the table from `reports` and your log (processing_time_s, compute)
def total_seconds(log_rows):
    vals = [r.get("processing_time_s", "").strip() for r in log_rows]
    return str(sum(float(v) for v in vals if v)) if any(vals) else ""
by_path = {"generative": {"processing_time_s": total_seconds(rows),
                          "compute": "; ".join(dict.fromkeys(r.get("compute", "") for r in rows))}} if rows else {}
if reports:
    print(f"{'output':26s} {'median err':>10s} {'success':>8s} {'NCC':>6s} {'flicker':>8s} {'warp err':>9s} {'bg PSNR':>8s} {'leakage':>8s} {'coverage':>9s} {'time s':>7s} {'compute':>8s}")
    for name, r in reports.items():
        p, f, t, s, o = r["placement"], r["fidelity"], r["temporal"], r["preservation"], r["occlusion"]
        g = lambda v, fmt: fmt.format(v) if v is not None and np.isfinite(v) else "n/a"
        lg = by_path.get(name, {})
        print(f"{name:26s} {g(p['median'], '{:9.2%}')} {g(p['success'], '{:8.0%}')} {g(f.get('ncc', float('nan')), '{:6.3f}')} {g(t['flicker'], '{:8.4f}')} {g(t['warping_error'], '{:9.4f}')} "
              f"{g(s.get('psnr_outside', float('nan')), '{:7.1f}')} {g(o['leakage'], '{:8.1%}')} {g(o['coverage_visible'], '{:9.1%}')} {lg.get('processing_time_s', ''):>7s} {lg.get('compute', ''):>8s}")

## (Our Code) Generative comparison helpers for both clips [AI-CODE] [HUMAN-CHECK]

We check our preserved inputs, extract frames from each run at nominal source timestamps, and build the source-reencode preservation baseline. We then format the comparison figures and table, using the classical selected-frame reports from Part I.

In [ ]:
"""Import both recorded attempts per student and reproduce the Stage 4 comparison."""

RECORDS = HERE / "run_records"
GENERATIVE_CLIPS = {clip: settings[0] for clip, settings in CLASSICAL_CLIPS.items()}


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=float) + "\n")


def extract_frames(video, indices, destination, original_size, output_fps=None, source_fps=30, start=0):
    destination.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    try:
        for f in indices:
            k = round((f-start)*output_fps/source_fps) if output_fps else f
            cap.set(cv2.CAP_PROP_POS_FRAMES, k)
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError(f"Cannot extract output index {k} for source frame {f}.")
            if (frame.shape[1], frame.shape[0]) != original_size:
                frame = cv2.resize(frame, original_size, interpolation=cv2.INTER_CUBIC)
            cv2.imwrite(str(destination / f"frame_{f:06d}.png"), frame)
    finally:
        cap.release()

def load_inputs(clip, entries):
    folder = CODE / "outputs/genai" / clip / "inputs"
    entry = next(e for e in entries if e["clip_id"] == clip and e["role"] == "corrected")
    run = json.loads((CODE / entry["records"] / "run.json").read_text())
    assert run["thresholds_sha256"] == digest(CODE / "outputs/thresholds.json")
    assert run["pipeline_stats_sha256"] == digest(CODE / "outputs/videos" / f"{clip}_flow_assisted_stats.json")
    for name, expected in run["assets_sha256"].items():
        assert digest(folder / name) == expected
    return run

def source_baseline(clip, run, chosen):
    video, _ = data.clip_paths(clip)
    folder = HERE / "source_baseline" / clip
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "reencoded.mp4"
    wanted = set(run["sample_source_indices"])
    with sp.VideoWriter(path, 16, (832, 480)) as writer:
        for index, frame in sp.iter_video(video, start=min(wanted), stop=max(wanted) + 1):
            if index in wanted:
                writer.write(cv2.resize(frame, (832, 480), interpolation=cv2.INTER_LANCZOS4))
    extract_frames(path, chosen, folder, (1280, 720), 16, 30, run["source"]["interval_start"])
    return folder

def comparison_figure(clip, conditions, original_folder, corrected_folder):
    video, _ = data.clip_paths(clip)
    gt = data.load_audit_corners(clip)
    named = [(label, conditions["frames"][label]) for label in ("clean", "occluded", "perspective")]
    source = {i: f for i, f in sp.iter_video(video, stop=max(f for _, f in named) + 1)
              if i in {f for _, f in named}}
    fig, axes = plt.subplots(3, 4, figsize=(16, 9))
    for row, (label, index) in enumerate(named):
        paths = [None, HERE / "comparison_frames" / clip / "classical_flow_assisted" / f"frame_{index:06d}.png",
                 original_folder / f"frame_{index:06d}.png", corrected_folder / f"frame_{index:06d}.png"]
        for col, path in enumerate(paths):
            image = source[index] if path is None else sp.read_image(path)
            axes[row, col].imshow(image)
            axes[row, col].set_title(f"{label}, source {index}" if col == 0 else
                                     ("Classical flow-assisted", "Original VACE", "Corrected VACE")[col - 1])
            q = gt[index]
            axes[row, col].plot(*np.vstack([q, q[0]]).T, color="lime", linewidth=.6)
            axes[row, col].set_xlim(-.5, 1279.5)
            axes[row, col].set_ylim(719.5, -.5)
            axes[row, col].axis("off")
    fig.suptitle(f"{clip}: fixed audit frames at nominal timestamps; green is the source audit quadrilateral")
    fig.subplots_adjust(left=.015, right=.985, bottom=.025, top=.91, wspace=.045, hspace=.24)
    fig.savefig(HERE / f"{clip}_generative_comparison.png", dpi=160)
    plt.close(fig)

def fmt(value, pattern=".3f"):
    return format(value, pattern) if value is not None and np.isfinite(value) else "n/a"

def comparison_markdown(results):
    lines = ["| Clip / method | Detected | Median error | NCC | Background PSNR | Leakage | Coverage (n) | Flicker | Warp error | Time (s) |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for clip, result in results.items():
        for method in ("classical_independent", "classical_flow_assisted", "generative_original", "generative"):
            report = result["reports"][method]
            p, f, s, o, t = [report[k] for k in ("placement", "fidelity", "preservation", "occlusion", "temporal")]
            lines.append(f"| {clip} / {method} | {p['n']-p['failed']}/{p['n']} | {fmt(p['median'], '.2%')} | "
                         f"{fmt(f['ncc'])} | {fmt(s['psnr_outside'], '.2f')} | {fmt(o['leakage'], '.2%')} | "
                         f"{fmt(o['coverage_visible'], '.2%')} ({o['n_coverage_frames']}) | {fmt(t['flicker'], '.4f')} | "
                         f"{fmt(t['warping_error'], '.4f')} | {fmt(result['costs'][method]['seconds'], '.2f')} |")
    return "\n".join(lines)

## (Our Code) Complete comparison and saved report evidence [AI-CODE] [HUMAN-CHECK]

We evaluate the original and tuned VACE outputs for each clip and compare them with both classical variants on the same selected frames. We display the comparison table, frame/mask/pair denominators and Figures 4.5–4.6. We save the aggregate results to `outputs/evaluation/stage4_genai_reports.json` and the individual reports under `outputs/evaluation/`. Our full-audit Scorecards and Figures 4.1–4.4 are saved in Part I.

In [ ]:
def analyse_generative():
    entries = RUN_ENTRIES
    frozen = PipelineConfig.from_json((CODE / "outputs/thresholds.json").read_text())
    results = {}
    evaluation = CODE / "outputs/evaluation"
    for clip, art_name in GENERATIVE_CLIPS.items():
        run = load_inputs(clip, entries)
        folder = CODE / "outputs/genai" / clip
        conditions = json.loads((folder / "conditions.json").read_text())
        fr = conditions["frames"]
        chosen = sorted({fr[k] for k in ("clean", "occluded", "perspective")} | set(fr["pair"]))
        mapping = {f: round((f-run['source']['interval_start'])*16/30) for f in chosen}
        assert len(set(mapping.values())) == len(chosen)
        video, _ = data.clip_paths(clip)
        gt, masks, artwork = data.load_audit_corners(clip), data.load_audit_masks(clip), data.load_artwork(art_name)
        reports, costs = {}, {}
        for role in ("original", "corrected"):
            entry = next(e for e in entries if e["clip_id"] == clip and e["role"] == role)
            raw = CODE / entry["output"]
            assert digest(raw) == entry["output_sha256"]
            assert sp.video_info(raw) == {"frames": 81, "fps": 16., "width": 832, "height": 480}
            target = raw.parent
            extract_frames(raw, chosen, target, (1280, 720), 16, 30, run["source"]["interval_start"])
            method = "generative" if role == "corrected" else "generative_original"
            reports[method] = evaluate_frames([target / f"frame_{f:06d}.png" for f in chosen], video, artwork,
                gt_corners=gt, gt_masks=masks, success_threshold=frozen.localisation.success_threshold, verbose=False)
            suffix = "genai" if role == "corrected" else "genai_original"
            write_json(evaluation / f"{clip}_{suffix}_frames_report.json", reports[method])
            meta = json.loads((CODE / entry["records"] / "run_metadata.json").read_text())
            costs[method] = {"seconds": meta["elapsed_s"], "setup_seconds": meta["setup"]["model_setup_s"],
                             "compute": f"Google Colab {meta['setup']['gpu']}", "invocation_id": entry["invocation_id"]}
            if role == "corrected":
                corrected_folder = target
            else:
                original_folder = target
        for variant in ("independent", "flow_assisted"):
            method = f"classical_{variant}"
            saved = personal_classical[clip]["variants"][variant]
            reports[method] = saved["comparison_baseline"]
            costs[method] = {"seconds": saved["time_s"], "compute": "Local CPU, hardware not recorded",
                             "source_frames": 450}
        target = source_baseline(clip, run, chosen)
        reports["source_reencoded"] = evaluate_frames([target / f"frame_{f:06d}.png" for f in chosen], video, artwork,
            gt_corners=gt, gt_masks=masks, success_threshold=frozen.localisation.success_threshold, verbose=False)
        write_json(evaluation / f"{clip}_source_reencoded_frames_report.json", reports["source_reencoded"])
        results[clip] = {"conditions": conditions, "source_indices": chosen, "output_indices": mapping,
                         "reports": reports, "costs": costs, "attempts": [e for e in entries if e["clip_id"] == clip]}
        comparison_figure(clip, conditions, original_folder, corrected_folder)
    write_json(evaluation / "stage4_genai_reports.json", results)
    write_json(evaluation / "genai_frames_report.json", results[CLIP_ID]["reports"]["generative"])
    return results

all_personal_reports = analyse_generative()
display(Markdown(comparison_markdown(all_personal_reports)))
for personal_clip, result in all_personal_reports.items():
    print(personal_clip, "nominal output mapping", result["output_indices"])
    for method, report in result["reports"].items():
        o = report["occlusion"]
        print(method, "placement frames", report["frames_with_corners"],
              "mask frames", report["frames_with_masks"],
              "nonempty masks", o["mask_frames"]-o["empty_mask_frames"],
              "coverage count", o["n_coverage_frames"],
              "artwork undetected on mask frames", o["overlay_absent_frames"],
              "temporal pairs", report["temporal"]["pair_frames"])
    display(Image(filename=str(ROOT / "student_files/stage4" / f"{personal_clip}_generative_comparison.png")))

for item in reversed(starter_created_paths):
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink(missing_ok=True)
print("Saved the both-person aggregate and per-run reports in outputs/evaluation/, with report figures in student_files/stage4/.")

### The staff oracle: MiniMax-H3 on the common clip
`data/genai_oracle/` holds one run of the same edit by a 33-billion-parameter open video model on the common clip, scored
by the same evaluator. Compare its failure modes with your model's: what does a much larger model fix, and what does it still get
wrong (background re-rendering, timing, occlusion order)?

In [ ]:
oracle = data.DATA_DIR / "genai_oracle"
if (oracle / "summary.md").exists():
    print((oracle / "summary.md").read_text())
    sheet = cv2.cvtColor(cv2.imread(str(oracle / "contact_sheet.jpg")), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 16 * sheet.shape[0] / sheet.shape[1])); plt.imshow(sheet); plt.axis("off")
    plt.title("rows: audit frames 160 / 192 / 320; columns: original, MiniMax-H3"); plt.show()
else:
    print("data/genai_oracle/ not found")

## 6. Reflection prompts for Question 5
Write the reflection from the table and the frames themselves. Classical versus generative on **your** clip along:

1. **Geometric placement** — does the artwork sit on the board's quadrilateral (median corner error, success) on the clean
   frame and on the strongest-perspective frame? Which method respects the keystone?
2. **Artwork identity** — is the artwork reproduced exactly (rectified NCC / SSIM / structure)? Where does the generative
   output drift (text, colours, layout)?
3. **Occlusion order** — on the occluded frame, does the occluder stay in front (leakage) and is the visible board painted
   (coverage)? Compare with your Stage 3 mask.
4. **Background preservation** — what does the background PSNR / SSIM / drift tell you about whether the pixels outside the
   board were *generated* (re-rendered frame) or *composited by code* (untouched)? Which method can you prove did not touch
   the background?
5. **Controllability** — what could you change by editing the prompt, what only by editing code?
6. **Timing** — does the output keep the clip's timing, or do the extracted audit frames no longer match their source frames
   (flicker and warping error on the consecutive pair, background drift)?
7. **Cost** — processing time, compute and manual effort, against the classical pipeline's run time.

Every claim must point to a row of the table or to a frame in `outputs/genai/`; mark generated pixels `[AI-GENERATION]`.

## Analysis ideas
* Plot corner error against mean flow magnitude for your clip; group frames by whether the monitor fired.
* Compare the two temporal variants frame by frame around the occlusion crossing.
* For the generative outputs, show one frame where the artwork identity drifts and quantify it with the rectified NCC.
* Treat the generation as a qualitative probe: one attempt says what the model *can* do, not how often.